In [5]:
# CELL 1 — MISTRAL EXPERIMENT SETUP

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import torch

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "mistral7b"

RUN.mkdir(parents=True, exist_ok=True)

print("===== MISTRAL EXPERIMENT =====")
print("Model:", MODEL_ID)
print("Output:", RUN)

print("\n===== GPU =====")

if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))
    print(
        "✅ VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("❌ No GPU detected")

print("\n===== DATA =====")

for name in ["train.jsonl", "val.jsonl", "test.jsonl"]:
    path = ROOT / "data" / "processed" / name
    print(
        "✅" if path.exists() else "❌",
        name,
        path
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
===== MISTRAL EXPERIMENT =====
Model: mistralai/Mistral-7B-Instruct-v0.3
Output: /content/drive/MyDrive/slm-distillation/outputs/mistral7b

===== GPU =====
❌ No GPU detected

===== DATA =====
✅ train.jsonl /content/drive/MyDrive/slm-distillation/data/processed/train.jsonl
✅ val.jsonl /content/drive/MyDrive/slm-distillation/data/processed/val.jsonl
✅ test.jsonl /content/drive/MyDrive/slm-distillation/data/processed/test.jsonl


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ Still on CPU")

CUDA available: True
GPU: Tesla T4


In [2]:
# CELL 2 — DEPENDENCIES + HUGGING FACE ACCESS CHECK

import sys
import subprocess
import os

print("🔧 Installing/checking training packages...")

packages = [
    "transformers>=4.45,<5",
    "accelerate>=1.0",
    "peft>=0.13",
    "bitsandbytes>=0.46.1",
    "datasets>=3.0",
    "huggingface_hub>=0.25",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + packages,
    check=True
)

print("✅ Packages ready")

# ------------------------------------------------------------
# Hugging Face token
# ------------------------------------------------------------

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ HF_TOKEN found")
else:
    print("❌ HF_TOKEN not found in Colab Secrets")

# ------------------------------------------------------------
# Check model access
# ------------------------------------------------------------

from huggingface_hub import HfApi

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

try:
    api = HfApi(token=HF_TOKEN)
    info = api.model_info(MODEL_ID)

    print("\n===== MODEL ACCESS =====")
    print("✅ Model found:", info.id)
    print("✅ Access check passed")

except Exception as e:
    print("\n❌ Model access problem:")
    print(type(e).__name__, str(e))

# ------------------------------------------------------------
# GPU check again
# ------------------------------------------------------------

import torch

print("\n===== GPU =====")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))

🔧 Installing/checking training packages...
✅ Packages ready
✅ HF_TOKEN found

===== MODEL ACCESS =====
✅ Model found: mistralai/Mistral-7B-Instruct-v0.3
✅ Access check passed

===== GPU =====
CUDA available: True
✅ GPU: Tesla T4


In [3]:
# CELL 3 — INSPECT TRAINING PROMPT / DATA FORMAT

import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/slm-distillation")
train_file = ROOT / "data" / "processed" / "train.jsonl"

print("Training file:", train_file)
print("Exists:", train_file.exists())

print("\n===== FIRST 2 TRAINING EXAMPLES =====")

with open(train_file, "r", encoding="utf-8") as f:
    for i in range(2):
        row = json.loads(f.readline())

        print(f"\n----- EXAMPLE {i+1} -----")
        print("Keys:", list(row.keys()))
        print(json.dumps(row, indent=2, ensure_ascii=False))

Training file: /content/drive/MyDrive/slm-distillation/data/processed/train.jsonl
Exists: False

===== FIRST 2 TRAINING EXAMPLES =====


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/slm-distillation/data/processed/train.jsonl'

In [4]:
# CELL 3 — REMOUNT DRIVE + INSPECT TRAINING DATA

from google.colab import drive
from pathlib import Path
import json

# Runtime restarted when GPU was enabled, so mount Drive again
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/slm-distillation")
train_file = ROOT / "data" / "processed" / "train.jsonl"

print("\n===== DRIVE CHECK =====")
print("Project folder exists:", ROOT.exists())
print("Training file exists:", train_file.exists())

if not train_file.exists():
    print("\n❌ train.jsonl still not found")
    print("We will locate the correct folder before doing anything else.")

else:
    print("\n===== FIRST 2 TRAINING EXAMPLES =====")

    with open(train_file, "r", encoding="utf-8") as f:
        for i in range(2):
            row = json.loads(f.readline())

            print(f"\n----- EXAMPLE {i+1} -----")
            print("Keys:", list(row.keys()))
            print(json.dumps(row, indent=2, ensure_ascii=False))


Mounted at /content/drive

===== DRIVE CHECK =====
Project folder exists: True
Training file exists: True

===== FIRST 2 TRAINING EXAMPLES =====

----- EXAMPLE 1 -----
Keys: ['text', 'cluster_id', 'prompt_id']
{
  "text": "<|im_start|>system\nYou are an expert at analyzing IT, HR, and customer experience support tickets. Your task is to generate a concise, descriptive label for a cluster of support tickets. The label should capture the dominant theme, be specific enough to distinguish this cluster from others, and be 5-15 words long. Return only the label — no explanation, no punctuation at the end, no quotes.<|im_end|>\n<|im_start|>user\nGenerate a concise label (5-15 words) for this cluster of support tickets.\n\nTicket 1: I need assistance to see when will my article arrive\nTicket 2: I have to check when my package is going to arrive\nTicket 3: can you show me when my product is going to arrive?\nTicket 4: I want to see how long it tyakes for my package to arrive\nTicket 5: I want 

In [5]:
# CELL 4 — INSPECT ALL PROMPT VARIANTS + DATASET COUNTS

import json
import re
from pathlib import Path
from collections import Counter, defaultdict

ROOT = Path("/content/drive/MyDrive/slm-distillation")
DATA = ROOT / "data" / "processed"

files = {
    "train": DATA / "train.jsonl",
    "val": DATA / "val.jsonl",
    "test": DATA / "test.jsonl",
}

all_rows = {}

for split, path in files.items():
    with open(path, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]

    all_rows[split] = rows

    print(f"{split.upper()}: {len(rows)} examples")
    print("Prompt IDs:", dict(Counter(r["prompt_id"] for r in rows)))
    print()

print("=" * 70)
print("UNIQUE TRAINING PROMPTS")
print("=" * 70)

prompt_examples = {}

for row in all_rows["train"]:
    pid = row["prompt_id"]

    if pid in prompt_examples:
        continue

    text = row["text"]

    user_match = re.search(
        r"<\|im_start\|>user\n(.*?)<\|im_end\|>",
        text,
        re.DOTALL,
    )

    if user_match:
        user_text = user_match.group(1).strip()

        # Show only the instruction before Ticket 1
        instruction = user_text.split("\n\nTicket 1:")[0].strip()

        prompt_examples[pid] = instruction

for pid in sorted(prompt_examples):
    print(f"\n{pid}:")
    print(prompt_examples[pid])

print("\n✅ Prompt inspection complete")

TRAIN: 70 examples
Prompt IDs: {'P1': 14, 'P2': 14, 'P3': 14, 'P4': 14, 'P5': 14}

VAL: 15 examples
Prompt IDs: {'P1': 3, 'P2': 3, 'P3': 3, 'P4': 3, 'P5': 3}

TEST: 20 examples
Prompt IDs: {'P1': 4, 'P2': 4, 'P3': 4, 'P4': 4, 'P5': 4}

UNIQUE TRAINING PROMPTS

P1:
Generate a concise label (5-15 words) for this cluster of support tickets.

P2:
What is the primary issue described in these support tickets? Answer in one short phrase.

P3:
Name this cluster as it would appear as a category in an IT support knowledge base.

P4:
Describe the common theme unifying these support tickets in 5-15 words.

P5:
Generate a label for this cluster that would help a chatbot route similar requests correctly.

✅ Prompt inspection complete


In [6]:
# CELL 5 — LOAD MISTRAL 7B IN 4-BIT

import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

print("🔄 Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=os.environ["HF_TOKEN"],
)

# Mistral does not always define a separate pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("✅ Tokenizer loaded")
print("Chat template available:", tokenizer.chat_template is not None)

# ------------------------------------------------------------
# 4-bit QLoRA configuration
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("\n🔄 Loading Mistral 7B in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=os.environ["HF_TOKEN"],
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.config.use_cache = False

print("\n✅ Mistral loaded successfully")

print("\n===== MODEL CHECK =====")
print("Model:", MODEL_ID)
print("Device:", next(model.parameters()).device)
print("4-bit:", getattr(model, "is_loaded_in_4bit", False))

print("\n===== GPU MEMORY =====")

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3

print("Allocated:", round(allocated, 2), "GB")
print("Reserved:", round(reserved, 2), "GB")
print(
    "Total:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

print("\n✅ Ready for LoRA setup")

🔄 Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]


✅ Mistral loaded successfully

===== MODEL CHECK =====
Model: mistralai/Mistral-7B-Instruct-v0.3
Device: cuda:0
4-bit: True

===== GPU MEMORY =====
Allocated: 3.86 GB
Reserved: 6.73 GB
Total: 14.56 GB

✅ Ready for LoRA setup


In [7]:
# CELL 6 — LORA SETUP + FORMAT PROJECT DATA FOR MISTRAL

import json
import re
import statistics
from pathlib import Path
from collections import Counter

import torch
from datasets import Dataset
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/slm-distillation")
DATA = ROOT / "data" / "processed"
RUN = ROOT / "outputs" / "mistral7b"

RUN.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Prepare Mistral for QLoRA
# ------------------------------------------------------------

print("🔧 Preparing Mistral for QLoRA...")

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

# Keep trainable LoRA parameters in FP32
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

model.config.use_cache = False

print("\n===== LORA CHECK =====")
model.print_trainable_parameters()

trainable_dtypes = {
    str(p.dtype)
    for p in model.parameters()
    if p.requires_grad
}

print("Trainable dtypes:", trainable_dtypes)

# ------------------------------------------------------------
# 2. Parse existing project ChatML examples
# ------------------------------------------------------------

def parse_project_example(row):
    text = row["text"]

    system_match = re.search(
        r"<\|im_start\|>system\n(.*?)<\|im_end\|>",
        text,
        re.DOTALL,
    )

    user_match = re.search(
        r"<\|im_start\|>user\n(.*?)<\|im_end\|>",
        text,
        re.DOTALL,
    )

    assistant_match = re.search(
        r"<\|im_start\|>assistant\n(.*?)<\|im_end\|>",
        text,
        re.DOTALL,
    )

    if not user_match or not assistant_match:
        raise ValueError("Could not parse training example")

    return {
        "system": system_match.group(1).strip() if system_match else "",
        "user": user_match.group(1).strip(),
        "assistant": assistant_match.group(1).strip(),
        "cluster_id": row["cluster_id"],
        "prompt_id": row["prompt_id"],
    }


# ------------------------------------------------------------
# 3. Detect whether Mistral template supports system role
# ------------------------------------------------------------

test_messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Say hello."
    },
    {
        "role": "assistant",
        "content": "Hello"
    },
]

try:
    tokenizer.apply_chat_template(
        test_messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    SYSTEM_ROLE_SUPPORTED = True

except Exception:
    SYSTEM_ROLE_SUPPORTED = False


print("\n===== CHAT TEMPLATE =====")
print("Native system role supported:", SYSTEM_ROLE_SUPPORTED)


# ------------------------------------------------------------
# 4. Convert examples to Mistral chat format
# ------------------------------------------------------------

def format_for_mistral(parsed):

    if SYSTEM_ROLE_SUPPORTED:

        messages = [
            {
                "role": "system",
                "content": parsed["system"],
            },
            {
                "role": "user",
                "content": parsed["user"],
            },
            {
                "role": "assistant",
                "content": parsed["assistant"],
            },
        ]

    else:

        # Preserve the exact system instructions,
        # but merge them into the first user message.
        combined_user = (
            parsed["system"]
            + "\n\n"
            + parsed["user"]
        )

        messages = [
            {
                "role": "user",
                "content": combined_user,
            },
            {
                "role": "assistant",
                "content": parsed["assistant"],
            },
        ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return formatted_text


# ------------------------------------------------------------
# 5. Load train / val / test
# ------------------------------------------------------------

def load_split(filename):

    rows = []

    with open(DATA / filename, "r", encoding="utf-8") as f:

        for line in f:

            if not line.strip():
                continue

            original = json.loads(line)
            parsed = parse_project_example(original)

            rows.append({
                "text": format_for_mistral(parsed),
                "cluster_id": parsed["cluster_id"],
                "prompt_id": parsed["prompt_id"],
                "expected": parsed["assistant"],
                "user_prompt": parsed["user"],
                "system_prompt": parsed["system"],
            })

    return rows


train_rows = load_split("train.jsonl")
val_rows = load_split("val.jsonl")
test_rows = load_split("test.jsonl")

train_dataset = Dataset.from_list(train_rows)
val_dataset = Dataset.from_list(val_rows)
test_dataset = Dataset.from_list(test_rows)


print("\n===== DATASET CHECK =====")
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

print(
    "Train prompt IDs:",
    dict(Counter(train_dataset["prompt_id"]))
)

# ------------------------------------------------------------
# 6. Check token lengths before training
# ------------------------------------------------------------

lengths = []

for text in train_dataset["text"]:

    ids = tokenizer(
        text,
        add_special_tokens=False,
    )["input_ids"]

    lengths.append(len(ids))


print("\n===== TOKEN LENGTHS =====")
print("Min:", min(lengths))
print("Median:", int(statistics.median(lengths)))
print("Max:", max(lengths))

# ------------------------------------------------------------
# 7. Show one Mistral-formatted example
# ------------------------------------------------------------

print("\n===== FIRST FORMATTED EXAMPLE =====")
print(train_dataset[0]["text"])

print("\nExpected label:")
print(train_dataset[0]["expected"])

# ------------------------------------------------------------
# 8. Save experiment configuration
# ------------------------------------------------------------

config = {
    "model_id": MODEL_ID,
    "method": "4-bit QLoRA",
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    "train_examples": len(train_dataset),
    "val_examples": len(val_dataset),
    "test_examples": len(test_dataset),
    "system_role_supported": SYSTEM_ROLE_SUPPORTED,
    "min_train_tokens": min(lengths),
    "median_train_tokens": int(statistics.median(lengths)),
    "max_train_tokens": max(lengths),
}

config_file = RUN / "experiment_config.json"

with open(config_file, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("\n✅ Config saved:")
print(config_file)

print("\n✅ Ready for training configuration")

🔧 Preparing Mistral for QLoRA...

===== LORA CHECK =====
trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754
Trainable dtypes: {'torch.float32'}

===== CHAT TEMPLATE =====
Native system role supported: True

===== DATASET CHECK =====
Train: 70
Val: 15
Test: 20
Train prompt IDs: {'P1': 14, 'P2': 14, 'P3': 14, 'P4': 14, 'P5': 14}

===== TOKEN LENGTHS =====
Min: 165
Median: 191
Max: 241

===== FIRST FORMATTED EXAMPLE =====
<s>[INST] Generate a concise label (5-15 words) for this cluster of support tickets.

Ticket 1: I need assistance to see when will my article arrive
Ticket 2: I have to check when my package is going to arrive
Ticket 3: can you show me when my product is going to arrive?
Ticket 4: I want to see how long it tyakes for my package to arrive
Ticket 5: I want assistance seeing how soon can I expect my parcel
Ticket 6: I need to check when my parcel is going to arrive
Ticket 7: I have to see how long it takes for my product to arrive
Ticket 8: I ha

In [8]:
# CELL 7 — FIX MISTRAL FORMAT SO SYSTEM PROMPT IS PRESERVED

import json
import re
import statistics
from pathlib import Path
from datasets import Dataset

ROOT = Path("/content/drive/MyDrive/slm-distillation")
DATA = ROOT / "data" / "processed"
RUN = ROOT / "outputs" / "mistral7b"

# ------------------------------------------------------------
# Parse original project example
# ------------------------------------------------------------

def parse_project_example(row):
    text = row["text"]

    system_match = re.search(
        r"<\|im_start\|>system\n(.*?)<\|im_end\|>",
        text,
        re.DOTALL,
    )

    user_match = re.search(
        r"<\|im_start\|>user\n(.*?)<\|im_end\|>",
        text,
        re.DOTALL,
    )

    assistant_match = re.search(
        r"<\|im_start\|>assistant\n(.*?)<\|im_end\|>",
        text,
        re.DOTALL,
    )

    if not user_match or not assistant_match:
        raise ValueError("Could not parse example")

    return {
        "system": system_match.group(1).strip() if system_match else "",
        "user": user_match.group(1).strip(),
        "assistant": assistant_match.group(1).strip(),
        "cluster_id": row["cluster_id"],
        "prompt_id": row["prompt_id"],
    }


# ------------------------------------------------------------
# Mistral formatting
#
# Merge system instructions into user message.
# This guarantees the original project instruction is preserved.
# ------------------------------------------------------------

def format_for_mistral(parsed):

    combined_user = (
        parsed["system"]
        + "\n\n"
        + parsed["user"]
    )

    messages = [
        {
            "role": "user",
            "content": combined_user,
        },
        {
            "role": "assistant",
            "content": parsed["assistant"],
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


# ------------------------------------------------------------
# Load all splits
# ------------------------------------------------------------

def load_split(filename):

    rows = []

    with open(DATA / filename, "r", encoding="utf-8") as f:

        for line in f:

            if not line.strip():
                continue

            original = json.loads(line)
            parsed = parse_project_example(original)

            rows.append({
                "text": format_for_mistral(parsed),
                "cluster_id": parsed["cluster_id"],
                "prompt_id": parsed["prompt_id"],
                "expected": parsed["assistant"],
                "user_prompt": parsed["user"],
                "system_prompt": parsed["system"],
            })

    return rows


train_rows = load_split("train.jsonl")
val_rows = load_split("val.jsonl")
test_rows = load_split("test.jsonl")

train_dataset = Dataset.from_list(train_rows)
val_dataset = Dataset.from_list(val_rows)
test_dataset = Dataset.from_list(test_rows)


# ------------------------------------------------------------
# Token lengths after preserving system prompt
# ------------------------------------------------------------

lengths = []

for text in train_dataset["text"]:
    ids = tokenizer(
        text,
        add_special_tokens=False,
    )["input_ids"]

    lengths.append(len(ids))


print("===== CORRECTED DATASET =====")
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

print("\n===== TOKEN LENGTHS =====")
print("Min:", min(lengths))
print("Median:", int(statistics.median(lengths)))
print("Max:", max(lengths))


# ------------------------------------------------------------
# Verify system prompt is actually present
# ------------------------------------------------------------

first_text = train_dataset[0]["text"]

system_preserved = (
    "You are an expert at analyzing IT, HR, and customer experience support tickets"
    in first_text
)

print("\nSystem prompt preserved:", system_preserved)

print("\n===== CORRECTED FIRST EXAMPLE =====")
print(first_text)

print("\nExpected label:")
print(train_dataset[0]["expected"])


# ------------------------------------------------------------
# Update experiment config
# ------------------------------------------------------------

config_file = RUN / "experiment_config.json"

if config_file.exists():
    with open(config_file, "r", encoding="utf-8") as f:
        config = json.load(f)
else:
    config = {}

config["prompt_format"] = "Mistral native chat template"
config["system_prompt_strategy"] = "merged_into_user_message"
config["system_prompt_preserved"] = system_preserved
config["min_train_tokens"] = min(lengths)
config["median_train_tokens"] = int(statistics.median(lengths))
config["max_train_tokens"] = max(lengths)

with open(config_file, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("\n✅ Updated config saved:")
print(config_file)

===== CORRECTED DATASET =====
Train: 70
Val: 15
Test: 20

===== TOKEN LENGTHS =====
Min: 250
Median: 276
Max: 326

System prompt preserved: True

===== CORRECTED FIRST EXAMPLE =====
<s>[INST] You are an expert at analyzing IT, HR, and customer experience support tickets. Your task is to generate a concise, descriptive label for a cluster of support tickets. The label should capture the dominant theme, be specific enough to distinguish this cluster from others, and be 5-15 words long. Return only the label — no explanation, no punctuation at the end, no quotes.

Generate a concise label (5-15 words) for this cluster of support tickets.

Ticket 1: I need assistance to see when will my article arrive
Ticket 2: I have to check when my package is going to arrive
Ticket 3: can you show me when my product is going to arrive?
Ticket 4: I want to see how long it tyakes for my package to arrive
Ticket 5: I want assistance seeing how soon can I expect my parcel
Ticket 6: I need to check when my p

In [9]:
# CELL 8 — MISTRAL BASELINE EVALUATION
# Save predictions BEFORE fine-tuning

import torch
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "mistral7b"

print("🔄 Running Mistral baseline on 20 test examples...\n")

# Faster inference settings
model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

baseline_results = []

# Disable LoRA adapters so this is the TRUE base Mistral model
with model.disable_adapter():

    for i, row in enumerate(test_rows, start=1):

        # Same project system prompt + prompt variation + tickets
        combined_user = (
            row["system_prompt"]
            + "\n\n"
            + row["user_prompt"]
        )

        messages = [
            {
                "role": "user",
                "content": combined_user
            }
        ]

        # Mistral-native inference prompt
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(
            prompt_text,
            return_tensors="pt",
            add_special_tokens=False,
        ).to("cuda")

        with torch.no_grad():

            outputs = model.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Decode ONLY newly generated tokens
        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

        prediction = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        ).strip()

        baseline_results.append({
            "example": i,
            "cluster_id": row["cluster_id"],
            "prompt_id": row["prompt_id"],
            "baseline_prediction": prediction,
            "expected": row["expected"],
        })

        print(f"{i:02d}/20 | {row['prompt_id']} | {prediction}")


# ------------------------------------------------------------
# Save baseline results
# ------------------------------------------------------------

baseline_df = pd.DataFrame(baseline_results)

baseline_file = RUN / "baseline_test_predictions.csv"

baseline_df.to_csv(
    baseline_file,
    index=False
)

print("\n==============================")
print("BASELINE COMPLETE")
print("==============================")

print("Examples:", len(baseline_df))

print("\n✅ Saved:")
print(baseline_file)


# ------------------------------------------------------------
# Restore training settings
# ------------------------------------------------------------

model.gradient_checkpointing_enable()
model.config.use_cache = False
model.train()

print("\n✅ Model restored to training mode")
print("✅ Ready for Mistral QLoRA training")

🔄 Running Mistral baseline on 20 test examples...

01/20 | P1 | Shipment Method Inquiry
02/20 | P2 | Shipment Options Inquiry
03/20 | P3 | Shipment Options Inquiry
04/20 | P4 | Shipment Method Inquiry
05/20 | P5 | ShipmentOptionsInquiry
06/20 | P1 | Sign-up Error Notifications
07/20 | P2 | Sign-up Errors Notification
08/20 | P3 | Sign-Up Error Notifications
09/20 | P4 | Sign-up Error Notifications
10/20 | P5 | Sign-up Error Notifications
11/20 | P1 | User PIN Recovery Requests
12/20 | P2 | User PIN Recovery Requests
13/20 | P3 | User Profile PIN Recovery
14/20 | P4 | User PIN Recovery Requests
15/20 | P5 | UserPINRecovery
16/20 | P1 | Gold Account Management
17/20 | P2 | Gold Account Management
18/20 | P3 | Gold Account Management
19/20 | P4 | Gold Account Management
20/20 | P5 | Gold Account Management

BASELINE COMPLETE
Examples: 20

✅ Saved:
/content/drive/MyDrive/slm-distillation/outputs/mistral7b/baseline_test_predictions.csv

✅ Model restored to training mode
✅ Ready for Mistral 

In [10]:
# CELL 9 — TRAIN MISTRAL 7B WITH QLoRA ON T4

import json
import gc
import torch
from pathlib import Path

from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "mistral7b"

TRAINING_DIR = RUN / "training"
FINAL_ADAPTER = RUN / "final_adapter"

TRAINING_DIR.mkdir(parents=True, exist_ok=True)
FINAL_ADAPTER.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 384

# ------------------------------------------------------------
# 1. Tokenize datasets
# ------------------------------------------------------------

print("🔄 Tokenizing training data...")

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        add_special_tokens=False,
    )


tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_val = val_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=val_dataset.column_names,
)

print("✅ Train:", len(tokenized_train))
print("✅ Val:", len(tokenized_val))


# ------------------------------------------------------------
# 2. Data collator
#
# For causal language modeling:
# labels = input_ids
# padding tokens are ignored automatically.
# ------------------------------------------------------------

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)


# ------------------------------------------------------------
# 3. Training preparation
# ------------------------------------------------------------

model.train()
model.config.use_cache = False
model.gradient_checkpointing_enable()

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# Keep LoRA parameters FP32
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# 4. Training configuration
#
# IMPORTANT:
# fp16=False / bf16=False prevents the AMP GradScaler issue
# we encountered during the earlier SmolLM2 experiment.
#
# The quantized Mistral layers still use FP16 compute.
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir=str(TRAINING_DIR),

    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    warmup_ratio=0.1,

    logging_steps=1,

    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=2,

    fp16=False,
    bf16=False,

    gradient_checkpointing=True,

    optim="adamw_torch",

    max_grad_norm=1.0,

    report_to="none",

    remove_unused_columns=False,

    seed=42,
)


# ------------------------------------------------------------
# 5. Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)


# ------------------------------------------------------------
# 6. Show final setup BEFORE training
# ------------------------------------------------------------

print("\n==============================")
print("MISTRAL TRAINING SETUP")
print("==============================")

print("Model:", MODEL_ID)
print("Train examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_val))
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Effective batch size:",
      training_args.per_device_train_batch_size
      * training_args.gradient_accumulation_steps)

print("Learning rate:", training_args.learning_rate)
print("Max sequence length:", MAX_LENGTH)

print("Trainer FP16:", training_args.fp16)
print("Trainer BF16:", training_args.bf16)

print(
    "GPU allocated before training:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "GPU reserved before training:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)


# ------------------------------------------------------------
# 7. TRAIN
# ------------------------------------------------------------

print("\n🚀 STARTING MISTRAL QLoRA TRAINING\n")

train_result = trainer.train()


# ------------------------------------------------------------
# 8. Save final adapter + tokenizer
# ------------------------------------------------------------

model.save_pretrained(FINAL_ADAPTER)
tokenizer.save_pretrained(FINAL_ADAPTER)


# ------------------------------------------------------------
# 9. Save metrics
# ------------------------------------------------------------

metrics = dict(train_result.metrics)

metrics["model_id"] = MODEL_ID
metrics["max_length"] = MAX_LENGTH
metrics["epochs"] = 3
metrics["learning_rate"] = 2e-4
metrics["train_batch_size"] = 1
metrics["gradient_accumulation_steps"] = 8

metrics_file = RUN / "training_metrics.json"

with open(metrics_file, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)


# ------------------------------------------------------------
# 10. Final validation loss
# ------------------------------------------------------------

print("\n🔄 Final validation evaluation...")

eval_metrics = trainer.evaluate()

eval_file = RUN / "validation_metrics.json"

with open(eval_file, "w", encoding="utf-8") as f:
    json.dump(eval_metrics, f, indent=2)


print("\n==============================")
print("✅ MISTRAL TRAINING COMPLETE")
print("==============================")

print("Final adapter:")
print(FINAL_ADAPTER)

print("\nTraining metrics:")
print(metrics)

print("\nValidation metrics:")
print(eval_metrics)

print("\n✅ Saved:")
print(metrics_file)
print(eval_file)

🔄 Tokenizing training data...


Map:   0%|          | 0/70 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

✅ Train: 70
✅ Val: 15

MISTRAL TRAINING SETUP
Model: mistralai/Mistral-7B-Instruct-v0.3
Train examples: 70
Validation examples: 15
Epochs: 3
Batch size: 1
Gradient accumulation: 8
Effective batch size: 8
Learning rate: 0.0002
Max sequence length: 384
Trainer FP16: False
Trainer BF16: False
GPU allocated before training: 4.52 GB
GPU reserved before training: 6.89 GB

🚀 STARTING MISTRAL QLoRA TRAINING



Epoch,Training Loss,Validation Loss
1,0.941000,0.844448
2,0.548700,0.809085
3,0.309100,0.896229



🔄 Final validation evaluation...



✅ MISTRAL TRAINING COMPLETE
Final adapter:
/content/drive/MyDrive/slm-distillation/outputs/mistral7b/final_adapter

Training metrics:
{'train_runtime': 392.4776, 'train_samples_per_second': 0.535, 'train_steps_per_second': 0.069, 'total_flos': 2506386908602368.0, 'train_loss': 0.8506239950656891, 'epoch': 3.0, 'model_id': 'mistralai/Mistral-7B-Instruct-v0.3', 'max_length': 384, 'epochs': 3, 'learning_rate': 0.0002, 'train_batch_size': 1, 'gradient_accumulation_steps': 8}

Validation metrics:
{'eval_loss': 0.8962286710739136, 'eval_runtime': 7.7359, 'eval_samples_per_second': 1.939, 'eval_steps_per_second': 1.939, 'epoch': 3.0}

✅ Saved:
/content/drive/MyDrive/slm-distillation/outputs/mistral7b/training_metrics.json
/content/drive/MyDrive/slm-distillation/outputs/mistral7b/validation_metrics.json


In [11]:
# CELL 10 — CHECK SAVED MISTRAL CHECKPOINTS

from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "mistral7b"
TRAINING_DIR = RUN / "training"

print("===== SAVED CHECKPOINTS =====")

checkpoints = sorted(
    [p for p in TRAINING_DIR.glob("checkpoint-*") if p.is_dir()],
    key=lambda p: int(p.name.split("-")[-1])
)

for checkpoint in checkpoints:
    print("✅", checkpoint)

print("\n===== VALIDATION LOSSES =====")
print("Epoch 1: 0.844448")
print("Epoch 2: 0.809085  <-- BEST")
print("Epoch 3: 0.896229")

best_checkpoint = TRAINING_DIR / "checkpoint-18"

print("\nBest checkpoint expected:")
print(best_checkpoint)

print("\nExists:", best_checkpoint.exists())

# Save the training conclusion now so we don't lose it
summary = {
    "model": "mistralai/Mistral-7B-Instruct-v0.3",
    "training_status": "successful",
    "epochs": 3,
    "train_loss": 0.8506239950656891,
    "validation_loss_epoch_1": 0.844448,
    "validation_loss_epoch_2": 0.809085,
    "validation_loss_epoch_3": 0.896229,
    "best_epoch": 2,
    "best_validation_loss": 0.809085,
    "best_checkpoint": str(best_checkpoint),
    "observation": "Validation loss improved through epoch 2 and increased at epoch 3, indicating mild overfitting."
}

summary_file = RUN / "training_summary.json"

with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\n✅ Training summary saved:")
print(summary_file)

===== SAVED CHECKPOINTS =====
✅ /content/drive/MyDrive/slm-distillation/outputs/mistral7b/training/checkpoint-18
✅ /content/drive/MyDrive/slm-distillation/outputs/mistral7b/training/checkpoint-27

===== VALIDATION LOSSES =====
Epoch 1: 0.844448
Epoch 2: 0.809085  <-- BEST
Epoch 3: 0.896229

Best checkpoint expected:
/content/drive/MyDrive/slm-distillation/outputs/mistral7b/training/checkpoint-18

Exists: True

✅ Training summary saved:
/content/drive/MyDrive/slm-distillation/outputs/mistral7b/training_summary.json


In [12]:
# CELL 11 — EVALUATE BEST FINE-TUNED MISTRAL CHECKPOINT

import torch
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "mistral7b"

BEST_CHECKPOINT = RUN / "training" / "checkpoint-18"
SAVE_FILE = RUN / "finetuned_test_predictions.csv"

print("===== BEST CHECKPOINT =====")
print(BEST_CHECKPOINT)
print("Exists:", BEST_CHECKPOINT.exists())

# ------------------------------------------------------------
# 1. Load epoch-2 adapter
# ------------------------------------------------------------

ADAPTER_NAME = "best_epoch2"

if ADAPTER_NAME not in model.peft_config:
    model.load_adapter(
        str(BEST_CHECKPOINT),
        adapter_name=ADAPTER_NAME
    )

model.set_adapter(ADAPTER_NAME)

print("✅ Best epoch-2 adapter loaded")

# ------------------------------------------------------------
# 2. Prepare inference
# ------------------------------------------------------------

model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

finetuned_results = []

print("\n🔄 Running fine-tuned Mistral on 20 test examples...\n")

# ------------------------------------------------------------
# 3. Run exact same test set
# ------------------------------------------------------------

for i, row in enumerate(test_rows, start=1):

    combined_user = (
        row["system_prompt"]
        + "\n\n"
        + row["user_prompt"]
    )

    messages = [
        {
            "role": "user",
            "content": combined_user,
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    finetuned_results.append({
        "example": i,
        "cluster_id": row["cluster_id"],
        "prompt_id": row["prompt_id"],
        "prediction": prediction,
        "expected": row["expected"],
    })

    print(
        f"{i:02d}/20 | "
        f"{row['prompt_id']} | "
        f"{prediction}"
    )

# ------------------------------------------------------------
# 4. Save predictions
# ------------------------------------------------------------

finetuned_df = pd.DataFrame(finetuned_results)

finetuned_df.to_csv(
    SAVE_FILE,
    index=False
)

print("\n==============================")
print("FINE-TUNED EVALUATION COMPLETE")
print("==============================")

print("Checkpoint: epoch 2 / checkpoint-18")
print("Examples:", len(finetuned_df))

print("\n✅ Saved:")
print(SAVE_FILE)

display(
    finetuned_df[
        [
            "example",
            "prompt_id",
            "prediction",
            "expected",
        ]
    ]
)

===== BEST CHECKPOINT =====
/content/drive/MyDrive/slm-distillation/outputs/mistral7b/training/checkpoint-18
Exists: True
✅ Best epoch-2 adapter loaded

🔄 Running fine-tuned Mistral on 20 test examples...

01/20 | P1 | Shipping options and methods inquiry and assistance requests
02/20 | P2 | Shipping options and methods availability and selection assistance requests
03/20 | P3 | Shipping Options and Methods Availability and Selection Assistance Requests
04/20 | P4 | Shipping options and methods availability and selection assistance requests
05/20 | P5 | Shipping options and methods inquiry and selection assistance requests
06/20 | P1 | Account creation problem reporting and escalation requests
07/20 | P2 | Account creation problem reporting and escalation process assistance requests
08/20 | P3 | Account Creation Problem Reporting and Escalation Procedures
09/20 | P4 | Account creation problem reporting and escalation process assistance requests
10/20 | P5 | Account creation problem rep

,example,prompt_id,prediction,expected
0,1,P1,Shipping options and methods inquiry and assis...,Inquiries about available shipping and deliver...
1,2,P2,Shipping options and methods availability and ...,Inquiry about available shipping and delivery ...
2,3,P3,Shipping Options and Methods Availability and ...,Shipping and Delivery Options Inquiry
3,4,P4,Shipping options and methods availability and ...,Customers requesting information about availab...
4,5,P5,Shipping options and methods inquiry and selec...,Shipping and delivery options inquiry
5,6,P1,Account creation problem reporting and escalat...,Sign-up process errors and account creation fa...
6,7,P2,Account creation problem reporting and escalat...,Sign-up process errors and notification assist...
7,8,P3,Account Creation Problem Reporting and Escalat...,Sign-up and Registration Error Reporting
8,9,P4,Account creation problem reporting and escalat...,Users seeking assistance reporting sign-up and...
9,10,P5,Account creation problem reporting and escalat...,Sign-up registration errors and problem reporting


In [13]:
# CELL 12 — MISTRAL BASELINE VS FINE-TUNED COSINE SIMILARITY

import sys
import subprocess
import pandas as pd
from pathlib import Path

# Make sure sentence-transformers is available
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"],
    check=True
)

from sentence_transformers import SentenceTransformer

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "mistral7b"

baseline_file = RUN / "baseline_test_predictions.csv"
finetuned_file = RUN / "finetuned_test_predictions.csv"

baseline = pd.read_csv(baseline_file)
finetuned = pd.read_csv(finetuned_file)

# Merge on example
df = finetuned.merge(
    baseline[
        [
            "example",
            "baseline_prediction"
        ]
    ],
    on="example"
)

print("🔄 Loading semantic similarity model...")

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# ------------------------------------------------------------
# Encode all text
# ------------------------------------------------------------

expected_embeddings = embedder.encode(
    df["expected"].tolist(),
    normalize_embeddings=True
)

baseline_embeddings = embedder.encode(
    df["baseline_prediction"].tolist(),
    normalize_embeddings=True
)

finetuned_embeddings = embedder.encode(
    df["prediction"].tolist(),
    normalize_embeddings=True
)

# Normalized vectors -> dot product = cosine similarity
df["baseline_similarity"] = (
    baseline_embeddings * expected_embeddings
).sum(axis=1)

df["finetuned_similarity"] = (
    finetuned_embeddings * expected_embeddings
).sum(axis=1)

df["improvement"] = (
    df["finetuned_similarity"]
    - df["baseline_similarity"]
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

baseline_avg = df["baseline_similarity"].mean()
finetuned_avg = df["finetuned_similarity"].mean()
avg_improvement = df["improvement"].mean()

better = (df["improvement"] > 0.0001).sum()
worse = (df["improvement"] < -0.0001).sum()
ties = len(df) - better - worse

print("\n==============================")
print("MISTRAL COSINE RESULTS")
print("==============================")

print(
    "Baseline average similarity:",
    round(baseline_avg, 4)
)

print(
    "Fine-tuned average similarity:",
    round(finetuned_avg, 4)
)

print(
    "Average improvement:",
    round(avg_improvement, 4)
)

print(
    f"Fine-tuned better on: {better}/{len(df)}"
)

print(
    f"Baseline better on: {worse}/{len(df)}"
)

print(
    f"Ties: {ties}/{len(df)}"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

save_file = RUN / "baseline_vs_finetuned_comparison.csv"

df.to_csv(
    save_file,
    index=False
)

print("\n✅ Saved comparison:")
print(save_file)

display(
    df[
        [
            "example",
            "prompt_id",
            "baseline_prediction",
            "prediction",
            "expected",
            "baseline_similarity",
            "finetuned_similarity",
            "improvement",
        ]
    ]
)

🔄 Loading semantic similarity model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


MISTRAL COSINE RESULTS
Baseline average similarity: 0.7673
Fine-tuned average similarity: 0.7318
Average improvement: -0.0355
Fine-tuned better on: 8/20
Baseline better on: 12/20
Ties: 0/20

✅ Saved comparison:
/content/drive/MyDrive/slm-distillation/outputs/mistral7b/baseline_vs_finetuned_comparison.csv


,example,prompt_id,baseline_prediction,prediction,expected,baseline_similarity,finetuned_similarity,improvement
0,1,P1,Shipment Method Inquiry,Shipping options and methods inquiry and assis...,Inquiries about available shipping and deliver...,0.621815,0.872062,0.250247
1,2,P2,Shipment Options Inquiry,Shipping options and methods availability and ...,Inquiry about available shipping and delivery ...,0.755765,0.769825,0.014060
2,3,P3,Shipment Options Inquiry,Shipping Options and Methods Availability and ...,Shipping and Delivery Options Inquiry,0.822996,0.698892,-0.124104
3,4,P4,Shipment Method Inquiry,Shipping options and methods availability and ...,Customers requesting information about availab...,0.508815,0.750548,0.241733
4,5,P5,ShipmentOptionsInquiry,Shipping options and methods inquiry and selec...,Shipping and delivery options inquiry,0.639324,0.798005,0.158681
5,6,P1,Sign-up Error Notifications,Account creation problem reporting and escalat...,Sign-up process errors and account creation fa...,0.825427,0.624989,-0.200438
6,7,P2,Sign-up Errors Notification,Account creation problem reporting and escalat...,Sign-up process errors and notification assist...,0.844937,0.486113,-0.358824
7,8,P3,Sign-Up Error Notifications,Account Creation Problem Reporting and Escalat...,Sign-up and Registration Error Reporting,0.738620,0.499216,-0.239404
8,9,P4,Sign-up Error Notifications,Account creation problem reporting and escalat...,Users seeking assistance reporting sign-up and...,0.618626,0.738613,0.119987
9,10,P5,Sign-up Error Notifications,Account creation problem reporting and escalat...,Sign-up registration errors and problem reporting,0.719608,0.547258,-0.172350


In [14]:
# CELL 13 — MISTRAL LLM-AS-A-JUDGE

import os
import json
import random
import time
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Anthropic key
# ------------------------------------------------------------

try:
    from google.colab import userdata

    if "ANTHROPIC_API_KEY" not in os.environ:
        os.environ["ANTHROPIC_API_KEY"] = userdata.get(
            "ANTHROPIC_API_KEY"
        )

except Exception:
    pass

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY not found in Colab Secrets"
    )

from anthropic import Anthropic

# ------------------------------------------------------------
# Files
# ------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "mistral7b"

comparison_file = RUN / "baseline_vs_finetuned_comparison.csv"

df = pd.read_csv(comparison_file)

client = Anthropic(
    api_key=os.environ["ANTHROPIC_API_KEY"]
)

random.seed(42)

judge_results = []

print("🤖 Running LLM judge on 20 Mistral examples...\n")

# ------------------------------------------------------------
# Judge each example
# ------------------------------------------------------------

for _, row in df.iterrows():

    # Randomize A/B to reduce position bias
    if random.random() < 0.5:

        candidate_a = row["baseline_prediction"]
        candidate_b = row["prediction"]

        a_type = "baseline"
        b_type = "finetuned"

    else:

        candidate_a = row["prediction"]
        candidate_b = row["baseline_prediction"]

        a_type = "finetuned"
        b_type = "baseline"

    prompt = f"""
You are evaluating two predicted cluster labels for customer support tickets.

REFERENCE LABEL:
{row["expected"]}

CANDIDATE A:
{candidate_a}

CANDIDATE B:
{candidate_b}

Judge which candidate is better based on:

1. Semantic similarity to the reference
2. Specificity to the support-ticket theme
3. Conciseness
4. Whether it works as a clean cluster label
5. Whether it follows the intended 5-15 word label style

Return ONLY valid JSON:

{{
  "winner": "A",
  "a_score": 1,
  "b_score": 1,
  "reason": "short reason"
}}

winner must be exactly:
A
B
or TIE

Scores must be integers from 1 to 5.
"""

    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=180,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    text = response.content[0].text.strip()

    try:

        start = text.index("{")
        end = text.rindex("}") + 1

        result = json.loads(
            text[start:end]
        )

        winner = result["winner"].upper()

        if winner == "A":
            actual_winner = a_type

        elif winner == "B":
            actual_winner = b_type

        else:
            actual_winner = "tie"

        judge_results.append({

            "example": int(row["example"]),

            "expected":
                row["expected"],

            "baseline_prediction":
                row["baseline_prediction"],

            "finetuned_prediction":
                row["prediction"],

            "winner":
                actual_winner,

            "baseline_score":
                result["a_score"]
                if a_type == "baseline"
                else result["b_score"],

            "finetuned_score":
                result["a_score"]
                if a_type == "finetuned"
                else result["b_score"],

            "reason":
                result["reason"],
        })

        print(
            f'{int(row["example"]):02d}/20 ✅ '
            f'Winner: {actual_winner}'
        )

    except Exception:

        print(
            f'{int(row["example"]):02d}/20 ❌ '
            f'Parse error: {text}'
        )

    time.sleep(0.2)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

judge_df = pd.DataFrame(
    judge_results
)

SAVE_FILE = RUN / "llm_judge_results.csv"

judge_df.to_csv(
    SAVE_FILE,
    index=False
)

finetuned_wins = (
    judge_df["winner"] == "finetuned"
).sum()

baseline_wins = (
    judge_df["winner"] == "baseline"
).sum()

ties = (
    judge_df["winner"] == "tie"
).sum()

baseline_score = (
    judge_df["baseline_score"].mean()
)

finetuned_score = (
    judge_df["finetuned_score"].mean()
)

print("\n==============================")
print("MISTRAL LLM JUDGE RESULTS")
print("==============================")

print(
    "Fine-tuned wins:",
    finetuned_wins
)

print(
    "Baseline wins:",
    baseline_wins
)

print(
    "Ties:",
    ties
)

print(
    "Baseline average score:",
    round(baseline_score, 3)
)

print(
    "Fine-tuned average score:",
    round(finetuned_score, 3)
)

print("\n✅ Saved:")
print(SAVE_FILE)

display(judge_df)

ModuleNotFoundError: No module named 'anthropic'

In [15]:
# INSTALL ANTHROPIC SDK

import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "anthropic"],
    check=True
)

from anthropic import Anthropic

print("✅ anthropic installed")

✅ anthropic installed


In [16]:
# CELL 13 — MISTRAL LLM-AS-A-JUDGE

import os
import json
import random
import time
import pandas as pd
from pathlib import Path
from anthropic import Anthropic

# ------------------------------------------------------------
# Load Anthropic API key
# ------------------------------------------------------------

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not ANTHROPIC_API_KEY:
    raise RuntimeError(
        "ANTHROPIC_API_KEY not found in Colab Secrets"
    )

client = Anthropic(
    api_key=ANTHROPIC_API_KEY
)

# ------------------------------------------------------------
# Load comparison file
# ------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "mistral7b"

comparison_file = RUN / "baseline_vs_finetuned_comparison.csv"

df = pd.read_csv(comparison_file)

random.seed(42)

judge_results = []

print("🤖 Running LLM judge on 20 Mistral examples...\n")

# ------------------------------------------------------------
# Judge each example
# ------------------------------------------------------------

for _, row in df.iterrows():

    # Randomize A/B to reduce position bias
    if random.random() < 0.5:
        candidate_a = row["baseline_prediction"]
        candidate_b = row["prediction"]

        a_type = "baseline"
        b_type = "finetuned"

    else:
        candidate_a = row["prediction"]
        candidate_b = row["baseline_prediction"]

        a_type = "finetuned"
        b_type = "baseline"

    prompt = f"""
You are evaluating two predicted cluster labels for customer support tickets.

REFERENCE LABEL:
{row["expected"]}

CANDIDATE A:
{candidate_a}

CANDIDATE B:
{candidate_b}

Judge which candidate is better based on:

1. Semantic similarity to the reference
2. Specificity to the support-ticket theme
3. Conciseness
4. Whether it works as a clean cluster label
5. Whether it follows the intended 5-15 word label style

Return ONLY valid JSON:

{{
  "winner": "A",
  "a_score": 1,
  "b_score": 1,
  "reason": "short reason"
}}

winner must be exactly A, B, or TIE.

Scores must be integers from 1 to 5.
"""

    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=180,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
    )

    text = response.content[0].text.strip()

    try:
        start = text.index("{")
        end = text.rindex("}") + 1

        result = json.loads(
            text[start:end]
        )

        winner = result["winner"].upper()

        if winner == "A":
            actual_winner = a_type

        elif winner == "B":
            actual_winner = b_type

        else:
            actual_winner = "tie"

        judge_results.append({
            "example": int(row["example"]),
            "expected": row["expected"],
            "baseline_prediction": row["baseline_prediction"],
            "finetuned_prediction": row["prediction"],
            "winner": actual_winner,

            "baseline_score":
                result["a_score"]
                if a_type == "baseline"
                else result["b_score"],

            "finetuned_score":
                result["a_score"]
                if a_type == "finetuned"
                else result["b_score"],

            "reason": result["reason"],
        })

        print(
            f'{int(row["example"]):02d}/20 ✅ '
            f'Winner: {actual_winner}'
        )

    except Exception as e:
        print(
            f'{int(row["example"]):02d}/20 ❌ '
            f'Parse error: {text}'
        )

    time.sleep(0.2)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

judge_df = pd.DataFrame(judge_results)

SAVE_FILE = RUN / "llm_judge_results.csv"

judge_df.to_csv(
    SAVE_FILE,
    index=False
)

finetuned_wins = (
    judge_df["winner"] == "finetuned"
).sum()

baseline_wins = (
    judge_df["winner"] == "baseline"
).sum()

ties = (
    judge_df["winner"] == "tie"
).sum()

baseline_score = (
    judge_df["baseline_score"].mean()
)

finetuned_score = (
    judge_df["finetuned_score"].mean()
)

print("\n==============================")
print("MISTRAL LLM JUDGE RESULTS")
print("==============================")

print(
    "Fine-tuned wins:",
    finetuned_wins
)

print(
    "Baseline wins:",
    baseline_wins
)

print(
    "Ties:",
    ties
)

print(
    "Baseline average score:",
    round(baseline_score, 3)
)

print(
    "Fine-tuned average score:",
    round(finetuned_score, 3)
)

print("\n✅ Saved:")
print(SAVE_FILE)

display(judge_df)

🤖 Running LLM judge on 20 Mistral examples...

01/20 ✅ Winner: finetuned
02/20 ✅ Winner: baseline
03/20 ✅ Winner: baseline
04/20 ✅ Winner: finetuned
05/20 ✅ Winner: finetuned
06/20 ✅ Winner: baseline
07/20 ✅ Winner: baseline
08/20 ✅ Winner: baseline
09/20 ✅ Winner: baseline
10/20 ✅ Winner: finetuned
11/20 ✅ Winner: baseline
12/20 ✅ Winner: finetuned
13/20 ✅ Winner: baseline
14/20 ✅ Winner: baseline
15/20 ✅ Winner: finetuned
16/20 ✅ Winner: finetuned
17/20 ✅ Winner: finetuned
18/20 ✅ Winner: finetuned
19/20 ✅ Winner: finetuned
20/20 ✅ Winner: baseline

MISTRAL LLM JUDGE RESULTS
Fine-tuned wins: 10
Baseline wins: 10
Ties: 0
Baseline average score: 3.4
Fine-tuned average score: 3.25

✅ Saved:
/content/drive/MyDrive/slm-distillation/outputs/mistral7b/llm_judge_results.csv


,example,expected,baseline_prediction,finetuned_prediction,winner,baseline_score,finetuned_score,reason
0,1,Inquiries about available shipping and deliver...,Shipment Method Inquiry,Shipping options and methods inquiry and assis...,finetuned,3,4,Candidate A has better semantic similarity ('a...
1,2,Inquiry about available shipping and delivery ...,Shipment Options Inquiry,Shipping options and methods availability and ...,baseline,4,2,"Candidate A is concise (3 words), clean, and s..."
2,3,Shipping and Delivery Options Inquiry,Shipment Options Inquiry,Shipping Options and Methods Availability and ...,baseline,4,2,"Candidate A is concise, semantically similar, ..."
3,4,Customers requesting information about availab...,Shipment Method Inquiry,Shipping options and methods availability and ...,finetuned,3,4,B better captures the full scope of the refere...
4,5,Shipping and delivery options inquiry,ShipmentOptionsInquiry,Shipping options and methods inquiry and selec...,finetuned,2,4,Candidate A matches the reference style (natur...
5,6,Sign-up process errors and account creation fa...,Sign-up Error Notifications,Account creation problem reporting and escalat...,baseline,4,3,"Candidate B is more concise (3 words vs 7), di..."
6,7,Sign-up process errors and notification assist...,Sign-up Errors Notification,Account creation problem reporting and escalat...,baseline,4,3,B is more concise (3 words vs 9) and closely m...
7,8,Sign-up and Registration Error Reporting,Sign-Up Error Notifications,Account Creation Problem Reporting and Escalat...,baseline,4,3,Candidate A better matches the reference label...
8,9,Users seeking assistance reporting sign-up and...,Sign-up Error Notifications,Account creation problem reporting and escalat...,baseline,4,3,Candidate A is more concise (3 words vs 11 wor...
9,10,Sign-up registration errors and problem reporting,Sign-up Error Notifications,Account creation problem reporting and escalat...,finetuned,3,4,Candidate B better captures both components of...


In [17]:
# CELL 14 — SHOW SAVED MISTRAL LLM JUDGE RESULTS

import pandas as pd
from pathlib import Path

RUN = Path("/content/drive/MyDrive/slm-distillation/outputs/mistral7b")

judge_file = RUN / "llm_judge_results.csv"

df = pd.read_csv(judge_file)

print("==============================")
print("MISTRAL LLM JUDGE RESULTS")
print("==============================")

print(
    "Fine-tuned wins:",
    (df["winner"] == "finetuned").sum()
)

print(
    "Baseline wins:",
    (df["winner"] == "baseline").sum()
)

print(
    "Ties:",
    (df["winner"] == "tie").sum()
)

print(
    "Baseline average score:",
    round(df["baseline_score"].mean(), 3)
)

print(
    "Fine-tuned average score:",
    round(df["finetuned_score"].mean(), 3)
)

MISTRAL LLM JUDGE RESULTS
Fine-tuned wins: 10
Baseline wins: 10
Ties: 0
Baseline average score: 3.4
Fine-tuned average score: 3.25


In [18]:
# CELL 15 — SAVE FINAL MISTRAL EXPERIMENT SUMMARY

import json
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "mistral7b"

comparison_file = RUN / "baseline_vs_finetuned_comparison.csv"
judge_file = RUN / "llm_judge_results.csv"
training_summary_file = RUN / "training_summary.json"

comparison = pd.read_csv(comparison_file)
judge = pd.read_csv(judge_file)

with open(training_summary_file, "r", encoding="utf-8") as f:
    training = json.load(f)

# ------------------------------------------------------------
# Calculate final metrics from saved files
# ------------------------------------------------------------

baseline_cosine = comparison["baseline_similarity"].mean()
finetuned_cosine = comparison["finetuned_similarity"].mean()
cosine_change = finetuned_cosine - baseline_cosine

finetuned_better = (comparison["improvement"] > 0.0001).sum()
baseline_better = (comparison["improvement"] < -0.0001).sum()

finetuned_wins = (judge["winner"] == "finetuned").sum()
baseline_wins = (judge["winner"] == "baseline").sum()
ties = (judge["winner"] == "tie").sum()

baseline_judge_score = judge["baseline_score"].mean()
finetuned_judge_score = judge["finetuned_score"].mean()

# ------------------------------------------------------------
# Create final summary
# ------------------------------------------------------------

summary = f"""# Mistral-7B-Instruct-v0.3 Benchmark Results

## Model

- Model: `mistralai/Mistral-7B-Instruct-v0.3`
- Fine-tuning method: 4-bit QLoRA
- GPU: Tesla T4
- LoRA rank: 16
- LoRA alpha: 16
- LoRA dropout: 0.05
- Learning rate: 2e-4
- Maximum sequence length: 384

## Dataset

- Training examples: 70
- Validation examples: 15
- Test examples: 20
- Prompt variants: P1, P2, P3, P4, P5
- Training epochs: 3

## Training

Training completed successfully.

- Final training loss: 0.8506
- Epoch 1 validation loss: 0.8444
- Epoch 2 validation loss: 0.8091
- Epoch 3 validation loss: 0.8962
- Best epoch: 2
- Best checkpoint: checkpoint-18

Validation performance improved through epoch 2 and declined at
epoch 3, suggesting mild overfitting.

## Cosine Similarity

- Baseline average similarity: {baseline_cosine:.4f}
- Fine-tuned average similarity: {finetuned_cosine:.4f}
- Change: {cosine_change:+.4f}
- Fine-tuned better on: {finetuned_better}/20
- Baseline better on: {baseline_better}/20

## LLM-as-a-Judge

- Fine-tuned wins: {finetuned_wins}
- Baseline wins: {baseline_wins}
- Ties: {ties}
- Baseline average score: {baseline_judge_score:.2f}
- Fine-tuned average score: {finetuned_judge_score:.2f}

## Conclusion

The baseline Mistral-7B-Instruct-v0.3 model performed slightly better
overall than the fine-tuned model in this smoke test.

Fine-tuning produced more detailed and descriptive labels, but the
baseline achieved higher average cosine similarity and a slightly
higher LLM-judge score.

This suggests that the base Mistral model was already strong for this
task and that the current QLoRA configuration did not provide an
overall improvement.

Future experiments could test different learning rates, fewer epochs,
LoRA parameters, or response-only loss masking.
"""

summary_file = RUN / "FINAL_MISTRAL_RESULTS.md"

summary_file.write_text(
    summary,
    encoding="utf-8"
)

# ------------------------------------------------------------
# Save machine-readable results too
# ------------------------------------------------------------

final_metrics = {
    "model": "mistralai/Mistral-7B-Instruct-v0.3",
    "training_status": "successful",
    "best_epoch": 2,
    "best_validation_loss": 0.809085,

    "baseline_cosine_similarity": float(baseline_cosine),
    "finetuned_cosine_similarity": float(finetuned_cosine),
    "cosine_change": float(cosine_change),

    "finetuned_better_examples": int(finetuned_better),
    "baseline_better_examples": int(baseline_better),

    "llm_judge_finetuned_wins": int(finetuned_wins),
    "llm_judge_baseline_wins": int(baseline_wins),
    "llm_judge_ties": int(ties),

    "baseline_judge_score": float(baseline_judge_score),
    "finetuned_judge_score": float(finetuned_judge_score),

    "overall_result": "baseline_performed_slightly_better"
}

metrics_file = RUN / "final_metrics.json"

with open(metrics_file, "w", encoding="utf-8") as f:
    json.dump(
        final_metrics,
        f,
        indent=2
    )

# ------------------------------------------------------------
# Show saved results
# ------------------------------------------------------------

print("==============================")
print("✅ MISTRAL EXPERIMENT SAVED")
print("==============================")

print("\nSummary:")
print(summary_file)

print("\nMetrics:")
print(metrics_file)

print("\n===== FINAL RESULT =====")
print("Baseline cosine:", round(baseline_cosine, 4))
print("Fine-tuned cosine:", round(finetuned_cosine, 4))
print("Baseline judge score:", round(baseline_judge_score, 2))
print("Fine-tuned judge score:", round(finetuned_judge_score, 2))

print("\nConclusion:")
print("Baseline Mistral performed slightly better overall.")

print("\n===== SAVED MISTRAL FILES =====")

for file in sorted(RUN.iterdir()):
    print("✅", file.name)

✅ MISTRAL EXPERIMENT SAVED

Summary:
/content/drive/MyDrive/slm-distillation/outputs/mistral7b/FINAL_MISTRAL_RESULTS.md

Metrics:
/content/drive/MyDrive/slm-distillation/outputs/mistral7b/final_metrics.json

===== FINAL RESULT =====
Baseline cosine: 0.7673
Fine-tuned cosine: 0.7318
Baseline judge score: 3.4
Fine-tuned judge score: 3.25

Conclusion:
Baseline Mistral performed slightly better overall.

===== SAVED MISTRAL FILES =====
✅ FINAL_MISTRAL_RESULTS.md
✅ baseline_test_predictions.csv
✅ baseline_vs_finetuned_comparison.csv
✅ experiment_config.json
✅ final_adapter
✅ final_metrics.json
✅ finetuned_test_predictions.csv
✅ llm_judge_results.csv
✅ training
✅ training_metrics.json
✅ training_summary.json
✅ validation_metrics.json


In [19]:
# FINAL SAFETY AUDIT — MISTRAL EXPERIMENT

from pathlib import Path
import os
import re
import subprocess

ROOT = Path("/content/drive/MyDrive/slm-distillation")
RUN = ROOT / "outputs" / "mistral7b"

print("=" * 60)
print("1. MISTRAL OUTPUT FILE CHECK")
print("=" * 60)

required_files = [
    "FINAL_MISTRAL_RESULTS.md",
    "final_metrics.json",
    "experiment_config.json",
    "baseline_test_predictions.csv",
    "finetuned_test_predictions.csv",
    "baseline_vs_finetuned_comparison.csv",
    "llm_judge_results.csv",
    "training_metrics.json",
    "training_summary.json",
    "validation_metrics.json",
]

all_good = True

for name in required_files:
    path = RUN / name

    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"✅ {name} ({size_kb:.1f} KB)")
    else:
        print(f"❌ MISSING: {name}")
        all_good = False


print("\n" + "=" * 60)
print("2. MODEL / CHECKPOINT CHECK")
print("=" * 60)

important_dirs = [
    RUN / "final_adapter",
    RUN / "training" / "checkpoint-18",
]

for path in important_dirs:
    print(
        "✅" if path.exists() else "❌",
        path
    )

best_checkpoint = RUN / "training" / "checkpoint-18"

print(
    "\nBest checkpoint safe:",
    best_checkpoint.exists()
)


print("\n" + "=" * 60)
print("3. LARGE FILE CHECK")
print("=" * 60)

print("These should stay in Google Drive and NOT go to GitHub:\n")

large_files = []

for path in RUN.rglob("*"):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)

        if size_mb > 20:
            large_files.append(
                (path, size_mb)
            )

if large_files:
    for path, size_mb in sorted(
        large_files,
        key=lambda x: x[1],
        reverse=True
    ):
        print(
            f"⚠️ {size_mb:.1f} MB | {path}"
        )
else:
    print("✅ No files over 20 MB found")


print("\n" + "=" * 60)
print("4. SECRET / API KEY SAFETY CHECK")
print("=" * 60)

# We only check text/result files.
# We DO NOT print any secret if one is found.

secret_patterns = {
    "Anthropic key": r"sk-ant-[A-Za-z0-9_\-]{10,}",
    "Hugging Face token": r"hf_[A-Za-z0-9]{10,}",
    "OpenAI-style key": r"sk-[A-Za-z0-9]{20,}",
}

text_extensions = {
    ".json",
    ".csv",
    ".md",
    ".txt",
    ".py",
}

secret_found = False

for path in RUN.rglob("*"):

    if (
        path.is_file()
        and path.suffix.lower() in text_extensions
        and path.stat().st_size < 10 * 1024 * 1024
    ):

        try:
            text = path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

            for secret_name, pattern in secret_patterns.items():

                if re.search(pattern, text):
                    print(
                        f"❌ Possible {secret_name} found in:"
                    )
                    print(path)
                    secret_found = True

        except Exception:
            pass

if not secret_found:
    print(
        "✅ No obvious API keys/tokens found in Mistral result files"
    )


print("\n" + "=" * 60)
print("5. FIND GITHUB REPOSITORY")
print("=" * 60)

repo = None

for root, dirs, files in os.walk("/content"):

    # Do not waste time scanning Drive/cache
    dirs[:] = [
        d for d in dirs
        if d not in {
            "drive",
            ".cache",
            "__pycache__"
        }
    ]

    if ".git" in dirs:
        repo = Path(root)
        break

if repo is None:
    print("⚠️ Git repository not currently found in /content")
    print("This does NOT affect your saved Drive results.")

else:
    print("✅ Repository:")
    print(repo)

    def git(*args):
        result = subprocess.run(
            ["git", "-C", str(repo), *args],
            capture_output=True,
            text=True
        )

        return result.stdout.strip()

    print("\nBranch:")
    print(git("branch", "--show-current"))

    print("\nGit status:")
    status = git("status", "--short")

    print(status if status else "✅ Clean working tree")


print("\n" + "=" * 60)
print("6. FINAL SAFETY RESULT")
print("=" * 60)

if all_good and not secret_found:
    print("✅ Core Mistral experiment files are safely saved")
    print("✅ No obvious secrets detected in result files")
    print("✅ Keep final_adapter/ and training/checkpoints in Drive")
    print("✅ Do NOT upload those large model files to GitHub")
else:
    print("⚠️ Review the warnings above BEFORE pushing to GitHub")

1. MISTRAL OUTPUT FILE CHECK
✅ FINAL_MISTRAL_RESULTS.md (1.6 KB)
✅ final_metrics.json (0.5 KB)
✅ experiment_config.json (0.6 KB)
✅ baseline_test_predictions.csv (1.6 KB)
✅ finetuned_test_predictions.csv (2.4 KB)
✅ baseline_vs_finetuned_comparison.csv (3.6 KB)
✅ llm_judge_results.csv (8.7 KB)
✅ training_metrics.json (0.4 KB)
✅ training_summary.json (0.5 KB)
✅ validation_metrics.json (0.1 KB)

2. MODEL / CHECKPOINT CHECK
✅ /content/drive/MyDrive/slm-distillation/outputs/mistral7b/final_adapter
✅ /content/drive/MyDrive/slm-distillation/outputs/mistral7b/training/checkpoint-18

Best checkpoint safe: True

3. LARGE FILE CHECK
These should stay in Google Drive and NOT go to GitHub:

⚠️ 320.4 MB | /content/drive/MyDrive/slm-distillation/outputs/mistral7b/training/checkpoint-18/optimizer.pt
⚠️ 320.4 MB | /content/drive/MyDrive/slm-distillation/outputs/mistral7b/training/checkpoint-27/optimizer.pt
⚠️ 160.1 MB | /content/drive/MyDrive/slm-distillation/outputs/mistral7b/final_adapter/adapter_mode

In [20]:
# FINAL NOTEBOOK SAFETY CHECK BEFORE GITHUB

from pathlib import Path
import re
import json

DRIVE = Path("/content/drive/MyDrive")
NOTEBOOK_NAME = "mistral7b_benchmark.ipynb"

print("=" * 60)
print("1. FIND NOTEBOOK")
print("=" * 60)

# Check common Colab location first
common_path = DRIVE / "Colab Notebooks" / NOTEBOOK_NAME

if common_path.exists():
    notebook = common_path
else:
    matches = list(DRIVE.rglob(NOTEBOOK_NAME))
    notebook = matches[0] if matches else None

if notebook is None:
    print("❌ Notebook not found")
    raise FileNotFoundError(NOTEBOOK_NAME)

print("✅ Found:")
print(notebook)

size_mb = notebook.stat().st_size / (1024 ** 2)
print(f"✅ Notebook size: {size_mb:.2f} MB")


print("\n" + "=" * 60)
print("2. NOTEBOOK VALIDITY CHECK")
print("=" * 60)

with open(notebook, "r", encoding="utf-8") as f:
    data = json.load(f)

print("✅ Valid .ipynb JSON")
print("✅ Cells:", len(data.get("cells", [])))


print("\n" + "=" * 60)
print("3. ACTUAL SECRET VALUE CHECK")
print("=" * 60)

raw = notebook.read_text(
    encoding="utf-8",
    errors="ignore"
)

# These search for actual token-shaped values,
# NOT harmless variable names such as HF_TOKEN.
patterns = {
    "Anthropic API key": r"sk-ant-[A-Za-z0-9_-]{15,}",
    "Hugging Face token": r"hf_[A-Za-z0-9]{15,}",
    "OpenAI-style API key": r"sk-[A-Za-z0-9_-]{20,}",
    "GitHub personal token": r"gh[pousr]_[A-Za-z0-9]{20,}",
}

found = []

for name, pattern in patterns.items():
    if re.search(pattern, raw):
        found.append(name)

if found:
    for name in found:
        print(f"❌ Possible {name} found")
else:
    print("✅ No obvious actual API keys/tokens found")


print("\n" + "=" * 60)
print("4. VARIABLE NAME CHECK")
print("=" * 60)

# Seeing these names is normal and SAFE.
for variable in [
    "HF_TOKEN",
    "ANTHROPIC_API_KEY"
]:
    if variable in raw:
        print(f"✅ {variable} variable name present — safe")
    else:
        print(f"ℹ️ {variable} variable name not present")


print("\n" + "=" * 60)
print("5. LARGE EMBEDDED OUTPUT CHECK")
print("=" * 60)

large_cells = []

for i, cell in enumerate(data.get("cells", [])):
    cell_size = len(
        json.dumps(cell)
    )

    if cell_size > 1_000_000:
        large_cells.append(
            (i, cell_size / 1024**2)
        )

if large_cells:
    for cell_num, mb in large_cells:
        print(
            f"⚠️ Cell {cell_num} contains about {mb:.2f} MB "
            "of notebook data/output"
        )
else:
    print("✅ No unusually huge notebook cells detected")


print("\n" + "=" * 60)
print("6. FINAL NOTEBOOK SAFETY RESULT")
print("=" * 60)

if not found:
    print("✅ Notebook is safe from obvious exposed secrets")
    print("✅ Notebook is valid")
    print("✅ Ready for GitHub packaging")
else:
    print("❌ DO NOT PUSH YET")
    print("Remove exposed credentials first")

1. FIND NOTEBOOK
✅ Found:
/content/drive/MyDrive/Colab Notebooks/mistral7b_benchmark.ipynb
✅ Notebook size: 0.35 MB

2. NOTEBOOK VALIDITY CHECK
✅ Valid .ipynb JSON
✅ Cells: 21

3. ACTUAL SECRET VALUE CHECK
✅ No obvious actual API keys/tokens found

4. VARIABLE NAME CHECK
✅ HF_TOKEN variable name present — safe
✅ ANTHROPIC_API_KEY variable name present — safe

5. LARGE EMBEDDED OUTPUT CHECK
✅ No unusually huge notebook cells detected

6. FINAL NOTEBOOK SAFETY RESULT
✅ Notebook is safe from obvious exposed secrets
✅ Notebook is valid
✅ Ready for GitHub packaging


In [21]:
# GITHUB STEP 1 — CLONE REPO + CHECK btt_dk BRANCH
# Does NOT commit or push anything yet.

import subprocess
from pathlib import Path
import shutil

REPO_URL = (
    "https://github.com/Break-Through-Tech/"
    "Automation-Anywhere-1A-domain-specific-theme-labeling-via-slm-distillation.git"
)

REPO = Path("/content/slm-distillation-repo")

# ------------------------------------------------------------
# 1. Remove old temporary clone if one exists
# ------------------------------------------------------------

if REPO.exists():
    print("🧹 Removing old temporary repo...")
    shutil.rmtree(REPO)

# ------------------------------------------------------------
# 2. Clone repository
# ------------------------------------------------------------

print("🔄 Cloning repository...")

result = subprocess.run(
    ["git", "clone", REPO_URL, str(REPO)],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("❌ Clone failed")
    print(result.stderr)
    raise RuntimeError("Git clone failed")

print("✅ Repository cloned")


# ------------------------------------------------------------
# 3. Fetch branches
# ------------------------------------------------------------

subprocess.run(
    ["git", "-C", str(REPO), "fetch", "--all"],
    check=True
)

# ------------------------------------------------------------
# 4. Check whether btt_dk exists remotely
# ------------------------------------------------------------

branches = subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "branch",
        "-a"
    ],
    capture_output=True,
    text=True,
    check=True
).stdout

print("\n===== BRANCHES =====")
print(branches)

if "remotes/origin/btt_dk" not in branches:
    raise RuntimeError(
        "❌ Remote branch origin/btt_dk was not found. "
        "Stop here and send me this output."
    )

# ------------------------------------------------------------
# 5. Checkout user's branch
# ------------------------------------------------------------

subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "checkout",
        "-B",
        "btt_dk",
        "origin/btt_dk"
    ],
    check=True
)

branch = subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "branch",
        "--show-current"
    ],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

print("\n✅ Current branch:", branch)


# ------------------------------------------------------------
# 6. Show latest commits
# ------------------------------------------------------------

history = subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "log",
        "--oneline",
        "-5"
    ],
    capture_output=True,
    text=True,
    check=True
).stdout

print("\n===== RECENT COMMITS =====")
print(history)


# ------------------------------------------------------------
# 7. Show top-level repository structure
# ------------------------------------------------------------

print("\n===== TOP-LEVEL REPO FILES =====")

for item in sorted(REPO.iterdir()):
    if item.name != ".git":
        marker = "📁" if item.is_dir() else "📄"
        print(marker, item.name)


# ------------------------------------------------------------
# 8. Git status
# ------------------------------------------------------------

status = subprocess.run(
    [
        "git",
        "-C",
        str(REPO),
        "status",
        "--short"
    ],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

print("\n===== GIT STATUS =====")

if status:
    print(status)
else:
    print("✅ Clean working tree")

print("\n✅ SAFE CHECK COMPLETE")
print("Nothing has been committed or pushed.")

🔄 Cloning repository...
✅ Repository cloned

===== BRANCHES =====
* main
  remotes/origin/Additional_Code_Refinements
  remotes/origin/Coach_experiments
  remotes/origin/Colab_Related_Code
  remotes/origin/HEAD -> origin/main
  remotes/origin/Initial_Code_Setup
  remotes/origin/Optimize_Trainer
  remotes/origin/Update_the_code
  remotes/origin/advisor_experiments
  remotes/origin/advisor_large_experiments
  remotes/origin/btt_dk
  remotes/origin/btt_ks
  remotes/origin/btt_setup_hn
  remotes/origin/btt_setup_mt
  remotes/origin/main
  remotes/origin/pair-X/model-name


✅ Current branch: btt_dk

===== RECENT COMMITS =====
2535508 docs: add SmolLM2 fine-tuning experiment results
2ed439c Update phase1_config.yaml
20b5f1e Update COLAB_SETUP_README.md
ee53b93 Update COLAB_SETUP_README.md
ee71125 Merge pull request #13 from Break-Through-Tech/Colab_Related_Code


===== TOP-LEVEL REPO FILES =====
📄 .gitignore
📄 COLAB_SETUP_README.md
📄 Challenge-Project-Overview.md
📄 EXPERIMENT_RESULTS.md
📄 Ge

In [22]:
# GITHUB STEP 2 — INSPECT REPO STRUCTURE
# Safe: does NOT copy, commit, or push anything.

from pathlib import Path
import subprocess

REPO = Path("/content/slm-distillation-repo")

print("=" * 70)
print("NOTEBOOKS")
print("=" * 70)

notebooks_dir = REPO / "notebooks"

if notebooks_dir.exists():
    for path in sorted(notebooks_dir.rglob("*")):
        if path.is_file():
            print(path.relative_to(REPO))
else:
    print("No notebooks directory found")


print("\n" + "=" * 70)
print("CODE")
print("=" * 70)

code_dir = REPO / "code"

if code_dir.exists():
    for path in sorted(code_dir.rglob("*")):
        if path.is_file():
            print(path.relative_to(REPO))
else:
    print("No code directory found")


print("\n" + "=" * 70)
print("GITIGNORE")
print("=" * 70)

gitignore = REPO / ".gitignore"

if gitignore.exists():
    print(gitignore.read_text(encoding="utf-8"))
else:
    print("No .gitignore found")


print("\n" + "=" * 70)
print("TRACKED RESULT / EXPERIMENT FILES")
print("=" * 70)

tracked = subprocess.run(
    ["git", "-C", str(REPO), "ls-files"],
    capture_output=True,
    text=True,
    check=True
).stdout.splitlines()

for file in tracked:
    lower = file.lower()

    if any(
        keyword in lower
        for keyword in [
            "experiment",
            "result",
            "benchmark",
            "evaluation",
            "metric"
        ]
    ):
        print(file)

print("\n✅ Inspection complete")
print("Nothing was modified.")

NOTEBOOKS
notebooks/.gitkeep
notebooks/00_session_setup.ipynb

CODE
code/CODE_README.md
code/configs/phase1_config.yaml
code/demo_tickets/cx_billing_dispute.txt
code/demo_tickets/demo_tickets.txt
code/demo_tickets/hr_new_hire_onboarding.txt
code/demo_tickets/it_password_reset.txt
code/demo_tickets/it_vpn_connectivity.txt
code/main.py
code/phase1/data/clustering.py
code/phase1/data/preprocessing.py
code/phase1/data/schema.py
code/phase1/evaluation/business_eval.py
code/phase1/evaluation/combine.py
code/phase1/evaluation/llm_judge.py
code/phase1/evaluation/metrics.py
code/phase1/finetuning/dataset.py
code/phase1/finetuning/trainer.py
code/phase1/inference_pipeline.py
code/phase1/labeling/frontier_llm.py
code/phase1/pipeline.py
code/phase1/prompts/templates.py
code/requirements.txt
code/requirements_colab.txt

GITIGNORE

code/outputs/
code/outputs/
outputs/
*.pth
*.safetensors
*.bin
__pycache__/
.DS_Store
venv/
.venv/
env/
**/venv/
**/.venv/
# Heavy folders & caches
/data/
outputs/
code/o